# 🛡️ Detecção de Fraudes em Cartão de Crédito com Machine Learning

Neste projeto, desenvolvemos um pipeline completo de Ciência de Dados para identificar transações fraudulentas em cartões de crédito utilizando dados reais anonimizados (`V1` a `V28` via PCA, além de `Time` e `Amount`).

### O Desafio do Desbalanceamento
Em bases de fraude financeira, apenas uma fração mínima das operações é ilícita (~0,17%). Um modelo ingênuo que classifique **todas** as transações como legítimas atinge **99,83% de acurácia**, mas falha completamente no objetivo de negócio (**0% de recall na classe de fraude**). Por isso, este estudo avalia os algoritmos pelas métricas da classe minoritária (`Class = 1`):
- **Recall (Sensibilidade):** Proporção de fraudes reais que o modelo conseguiu capturar.
- **Precision (Precisão):** Proporção de alertas gerados que eram realmente fraudes (evita bloquear clientes legítimos sem necessidade).
- **F1-Score & Curva Precision-Recall (PR-AUC):** Equilíbrio entre captura de fraudes e controle de falsos positivos.

## 1. Exploração dos Dados

Carregamos o dataset diretamente via URL pública utilizando `pandas` e inspecionamos a estrutura das 31 colunas e a proporção exata entre transações normais (`0`) e fraudulentas (`1`).

In [ ]:
import pandas as pd

url = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"

df = pd.read_csv(url)

df.head()

In [ ]:
proporcao = df['Class'].value_counts(normalize=True)
ir = proporcao.max() / proporcao.min()  # Imbalance Ratio

if proporcao.min() < 0.01:
    status = "extremamente desbalanceado"
elif proporcao.min() < 0.20:
    status = "moderadamente desbalanceado"
elif proporcao.min() < 0.40:
    status = "levemente desbalanceado"
else:
    status = "balanceado"

print(f"A saída indica que a class = 0 tem {proporcao[0]:.2%} e class = 1 tem {proporcao[1]:.3%}.")
print(f"Razão de desbalanceamento (IR): {ir:.0f}:1 — o dataset é {status}.")

## 2. Preparação dos Dados

Como as variáveis `V1` a `V28` já passaram por transformação PCA, tratamos a assimetria da coluna `Amount` aplicando transformação logarítmica (`np.log1p`) e padronização com `StandardScaler`. Em seguida, dividimos os dados em **70% treino e 30% teste** usando `stratify=y` para preservar a mesma taxa de 0,173% de fraudes em ambos os conjuntos.

In [ ]:
import numpy as np

df["amount_log"] = np.log1p(df['Amount'])

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])

In [ ]:
from sklearn.model_selection import train_test_split

x = df.drop(["Class"], axis=1)
y = df["Class"]

x_train, x_test, y_train, y_test = train_test_split(
    x, y, stratify=y, test_size=0.30, random_state=42
)

## 3. Treinamento e Avaliação Inicial dos Modelos

### 3.1 Baseline — Logistic Regression
Iniciamos com a Regressão Logística como referência inicial para medir o comportamento linear sobre os dados.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=2000)
model.fit(x_train, y_train)

y_pred = model.predict(x_test)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

Apesar da Acurácia alta (`1.00`), temos que nos atentar que isso não garante um bom resultado, já que podemos cair no **paradoxo da acurácia**. Por isso a importância de analisar outras métricas na classe `1`:

* **Recall**
* **F1-score**
* **Precision**

### 3.2 Curva ROC e Curva Precision-Recall (Baseline)
Em problemas altamente desbalanceados, a **Curva Precision-Recall** é mais informativa do que a Curva ROC, pois a ROC pode parecer otimista devido ao grande volume de verdadeiros negativos.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

y_probs = model.predict_proba(x_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_probs)

plt.plot(fpr, tpr)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.show

print("AUC", roc_auc_score(y_test, y_probs))

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recalls, _ = precision_recall_curve(y_test, y_probs)

plt.plot(recalls, precision)
plt.title("Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.show()

### 3.3 Random Forest (`class_weight='balanced'`)
Testamos um modelo baseado em árvores de decisão penalizando automaticamente os erros na classe minoritária através do parâmetro `class_weight="balanced"`.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators =50,
    max_depth = 10,
    random_state = 42,
    class_weight = "balanced",
    n_jobs = -1

)

rf.fit(x_train, y_train)

y_pred_rf = rf.predict(x_test)

print(classification_report(y_test, y_pred_rf))

y_pred = model.predict(x_test)

## 4. Pipeline, Ajuste de Limiar e XGBoost

### 4.1 Pipeline com Padronização + Regressão Logística e Limiar Customizado (`threshold = 0.3`)
Encapsulamos o `StandardScaler` e o classificador em um `Pipeline` (evitando vazamento de dados) e reduzimos o limiar de corte de `0.5` para `0.3` para aumentar a sensibilidade (recall) na detecção de fraudes.

In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2000))
])

pipeline.fit(x_train, y_train)

y_pred = pipeline.predict(x_test)

In [ ]:
threshold = 0.3

y_pred_custom = (y_probs >= threshold).astype(int)

print(classification_report(y_test, y_pred_custom))

### 4.2 XGBoost (`scale_pos_weight = 10`)
Treinamos o **XGBoost**, aplicando peso maior à classe positiva (`scale_pos_weight = 10`) para lidar com o desbalanceamento extremo.

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
   scale_pos_weight = 10,
   use_label_encoder = False,
   eval_metric = "logloss",
)

xgb.fit(x_train, y_train)

y_pred_xgb = xgb.predict(x_test)

print(classification_report(y_test, y_pred_xgb))

## 5. Otimização de Hiperparâmetros com `GridSearchCV`

Utilizamos validação cruzada (`cv=3`) otimizando diretamente pela métrica `scoring="recall"` para encontrar a melhor combinação entre número de árvores (`n_estimators`) e profundidade máxima (`max_depth`).

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [10, 50, 100,],
    'max_depth': [3, 5, 7]
}

grid_search = GridSearchCV(
    XGBClassifier(eval_metric='logloss'),
    param_grid,
    cv=3,
    scoring = "recall"
)

grid_search.fit(x_train, y_train)

print(grid_search.best_params_)

## 6. Explicabilidade do Modelo (Feature Importance e SHAP)

Para abrir a "caixa-preta" do XGBoost e justificar por que uma transação é marcada como fraude, analisamos:
1. **Feature Importances** nativo das árvores do XGBoost.
2. **Valores SHAP (SHapley Additive exPlanations)**, medindo o impacto médio absoluto de cada variável nas decisões do modelo.

In [ ]:
import matplotlib.pyplot as plt

importancias = xgb.feature_importances_

plt.bar(range(len(importancias)), importancias)
plt.title("Feature Importances")
plt.show()

In [ ]:
import shap

explainer = shap.Explainer(xgb)
shap_values = explainer(x_test[:100])
shap.plots.bar(shap_values)

## 7. Evolução do Desafio — Comparação de Balanceamento (Undersampling vs. Oversampling/SMOTE) e Pipeline com Random Forest

Seguindo as ideias de evolução do desafio:
1. **Comparação Undersampling vs. Oversampling (SMOTE):** Aplicamos `RandomUnderSampler` e `SMOTE` **exclusivamente sobre o conjunto de treino** (`x_train`, `y_train`) para evitar vazamento de dados no teste (`x_test`) e comparamos o impacto no *Recall* e na *Precision*.
2. **Pipeline com Random Forest + Ajuste de Limiar (`0.3`):** Encapsulamos o `StandardScaler` e o `RandomForestClassifier(class_weight='balanced')` em um `Pipeline` completo e avaliamos o ganho de *Recall* ao baixar o threshold para `0.3`.
3. **Curva Precision-Recall Comparativa:** Consolidamos todos os modelos principais em um único gráfico para visualizar o trade-off entre Precisão e Recall.

In [ ]:
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

# 1. Undersampling (reduz a classe majoritária no treino)
rus = RandomUnderSampler(random_state=42)
x_train_rus, y_train_rus = rus.fit_resample(x_train, y_train)

xgb_rus = XGBClassifier(eval_metric="logloss", random_state=42)
xgb_rus.fit(x_train_rus, y_train_rus)
y_pred_rus = xgb_rus.predict(x_test)

print("=== XGBoost com RandomUndersampling ===")
print(classification_report(y_test, y_pred_rus))

# 2. Oversampling com SMOTE (gera exemplos sintéticos de fraude no treino)
smote = SMOTE(random_state=42)
x_train_smote, y_train_smote = smote.fit_resample(x_train, y_train)

xgb_smote = XGBClassifier(eval_metric="logloss", random_state=42)
xgb_smote.fit(x_train_smote, y_train_smote)
y_pred_smote = xgb_smote.predict(x_test)

print("=== XGBoost com Oversampling (SMOTE) ===")
print(classification_report(y_test, y_pred_smote))

In [ ]:
# Pipeline com Random Forest + Ajuste de Limiar (0.3)
rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipeline.fit(x_train, y_train)
y_probs_rf_pipe = rf_pipeline.predict_proba(x_test)[:, 1]
y_pred_rf_03 = (y_probs_rf_pipe >= 0.3).astype(int)

print("=== Pipeline Random Forest (Threshold = 0.3) ===")
print(classification_report(y_test, y_pred_rf_03))

In [ ]:
from sklearn.metrics import average_precision_score

y_probs_xgb = xgb.predict_proba(x_test)[:, 1]
y_probs_smote = xgb_smote.predict_proba(x_test)[:, 1]
y_probs_rus = xgb_rus.predict_proba(x_test)[:, 1]

modelos_pr = [
    ("Logistic Regression (Baseline)", y_probs),
    ("Random Forest (Pipeline)", y_probs_rf_pipe),
    ("XGBoost (scale_pos_weight=10)", y_probs_xgb),
    ("XGBoost + SMOTE", y_probs_smote),
    ("XGBoost + Undersampling", y_probs_rus),
]

plt.figure(figsize=(9, 6))
for nome, probs in modelos_pr:
    prec, rec, _ = precision_recall_curve(y_test, probs)
    ap = average_precision_score(y_test, probs)
    plt.plot(rec, prec, label=f"{nome} (PR-AUC = {ap:.3f})")

plt.title("Curva Precision-Recall — Comparação Completa dos Modelos")
plt.xlabel("Recall (Fraudes Capturadas)")
plt.ylabel("Precision (Assertividade dos Alertas)")
plt.legend(loc="lower left")
plt.grid(True, alpha=0.3)
plt.show()

## 8. Conclusão Final

* **Por que não usar Acurácia:** Com apenas 0,173% de fraudes (`578:1`), todos os modelos atingem ~100% de acurácia. As decisões foram guiadas por **Recall**, **Precision**, **F1-score** e **PR-AUC** na classe `1`.
* **Undersampling vs. Oversampling (SMOTE):** O *Undersampling* maximiza o *Recall* (captura quase todas as fraudes), porém derruba a *Precision* por descartar grande parte dos exemplos legítimos no treino. Já o **SMOTE** e o **XGBoost ponderado (`scale_pos_weight=10`)** preservam a base original e oferecem o melhor equilíbrio operacional (alto F1-score e alta PR-AUC).
* **Interpretabilidade com SHAP:** As componentes `V14`, `V4`, `V12`, `V10` e `V17` são os principais sinais de alerta identificados pelo modelo para classificar uma operação como fraudulenta.